In [37]:
# Cell 1 - Import library

from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

print("Libraries loaded.")

Libraries loaded.


In [38]:
# Cell 2 - Konfigurasi dataset dan output audit

# ============================================================
# DATASET PATH CONFIG
# ============================================================

DEV_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/A.Raw Dataset")
TEST_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/A.Raw_Testing")

AUDIT_OUT_DIR = Path("/media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset")
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# DATASET STRUCTURE CONFIG
# ============================================================

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

DEV_SUBJECTS = [
    "Adelia",
    "Afi",
    "Aswangga",
    "Bustan",
    "Dilia",
    "Eldivo",
    "Fathir",
    "Lina",
    "Manda",
    "Miftah",
    "Teguh",
    "Tsamara",
]

TEST_ROOMS = ["Controlled Room", "Uncontrolled Room"]

TEST_SUBJECTS = [
    "Kanaya",
    "Naila",
    "Nana",
    "Rega",
    "Zaira",
]

EXPECTED_FILES = [f"{i}.Csv" for i in range(1, 10)]

# Kolom utama untuk audit similarity
REQUIRED_COLUMNS = ["Timestamp", "X", "Y", "Z"]

# ============================================================
# AUDIT CONFIG
# ============================================================

ROUND_DECIMALS_XYZ = 6

# Threshold status
TH_OK = 0.001
TH_WARNING = 0.01
TH_HIGH = 0.05

print("===== AUDIT CONFIG =====")
print(f"DEV_BASE_DIR  : {DEV_BASE_DIR}")
print(f"TEST_BASE_DIR : {TEST_BASE_DIR}")
print(f"AUDIT_OUT_DIR : {AUDIT_OUT_DIR}")
print(f"Activities    : {ACTIVITIES}")
print(f"Dev subjects  : {len(DEV_SUBJECTS)} subjects")
print(f"Test subjects : {len(TEST_SUBJECTS)} subjects")
print(f"Test rooms    : {TEST_ROOMS}")
print(f"Files         : {EXPECTED_FILES}")
print(f"Columns       : {REQUIRED_COLUMNS}")

===== AUDIT CONFIG =====
DEV_BASE_DIR  : /media/spell/Spell-lab/Lidar/A.Raw Dataset
TEST_BASE_DIR : /media/spell/Spell-lab/Lidar/A.Raw_Testing
AUDIT_OUT_DIR : /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset
Activities    : ['Bungkuk', 'Duduk', 'Jongkok', 'Jatuh']
Dev subjects  : 12 subjects
Test subjects : 5 subjects
Test rooms    : ['Controlled Room', 'Uncontrolled Room']
Files         : ['1.Csv', '2.Csv', '3.Csv', '4.Csv', '5.Csv', '6.Csv', '7.Csv', '8.Csv', '9.Csv']
Columns       : ['Timestamp', 'X', 'Y', 'Z']


In [39]:
# Cell 3 - Simpan konfigurasi audit

audit_config = {
    "DEV_BASE_DIR": str(DEV_BASE_DIR),
    "TEST_BASE_DIR": str(TEST_BASE_DIR),
    "AUDIT_OUT_DIR": str(AUDIT_OUT_DIR),
    "ACTIVITIES": ACTIVITIES,
    "DEV_SUBJECTS": DEV_SUBJECTS,
    "TEST_ROOMS": TEST_ROOMS,
    "TEST_SUBJECTS": TEST_SUBJECTS,
    "EXPECTED_FILES": EXPECTED_FILES,
    "REQUIRED_COLUMNS": REQUIRED_COLUMNS,
    "ROUND_DECIMALS_XYZ": ROUND_DECIMALS_XYZ,
    "TH_OK": TH_OK,
    "TH_WARNING": TH_WARNING,
    "TH_HIGH": TH_HIGH,
}

config_path = AUDIT_OUT_DIR / "audit_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(audit_config, f, indent=4, ensure_ascii=False)

print(f"Config saved to: {config_path}")

Config saved to: /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/audit_config.json


In [40]:
# Cell 4 - Buat manifest semua file yang diharapkan

manifest_records = []

# Development files
for activity in ACTIVITIES:
    for subject in DEV_SUBJECTS:
        for file_name in EXPECTED_FILES:
            file_path = DEV_BASE_DIR / activity / subject / "CSV" / file_name

            manifest_records.append({
                "split": "development",
                "room": "development",
                "activity": activity,
                "subject": subject,
                "file_name": file_name,
                "file_id": int(Path(file_name).stem),
                "file_path": str(file_path),
                "exists": file_path.exists(),
            })

# Testing files
for room in TEST_ROOMS:
    for activity in ACTIVITIES:
        for subject in TEST_SUBJECTS:
            for file_name in EXPECTED_FILES:
                file_path = TEST_BASE_DIR / room / activity / subject / "CSV" / file_name

                manifest_records.append({
                    "split": "testing",
                    "room": room,
                    "activity": activity,
                    "subject": subject,
                    "file_name": file_name,
                    "file_id": int(Path(file_name).stem),
                    "file_path": str(file_path),
                    "exists": file_path.exists(),
                })

manifest_df = pd.DataFrame(manifest_records)

manifest_path = AUDIT_OUT_DIR / "audit_manifest_all_files.csv"
manifest_df.to_csv(manifest_path, index=False)

print("===== DATASET MANIFEST =====")
print(f"Total expected files : {len(manifest_df):,}")
print(f"Existing files       : {manifest_df['exists'].sum():,}")
print(f"Missing files        : {(~manifest_df['exists']).sum():,}")
print(f"Manifest saved       : {manifest_path}")

display(manifest_df.head())
display(manifest_df[manifest_df["exists"] == False].head(20))

===== DATASET MANIFEST =====
Total expected files : 792
Existing files       : 792
Missing files        : 0
Manifest saved       : /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/audit_manifest_all_files.csv


,split,room,activity,subject,file_name,file_id,file_path,exists
0,development,development,Bungkuk,Adelia,1.Csv,1,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/1.Csv,True
1,development,development,Bungkuk,Adelia,2.Csv,2,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,True
2,development,development,Bungkuk,Adelia,3.Csv,3,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,True
3,development,development,Bungkuk,Adelia,4.Csv,4,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv,True
4,development,development,Bungkuk,Adelia,5.Csv,5,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv,True


,split,room,activity,subject,file_name,file_id,file_path,exists


In [41]:
# Cell 5 - Buat daftar pasangan file berurutan

pair_records = []

# Development adjacent pairs
for activity in ACTIVITIES:
    for subject in DEV_SUBJECTS:
        for i in range(1, 9):
            file_a = f"{i}.Csv"
            file_b = f"{i+1}.Csv"

            path_a = DEV_BASE_DIR / activity / subject / "CSV" / file_a
            path_b = DEV_BASE_DIR / activity / subject / "CSV" / file_b

            pair_records.append({
                "split": "development",
                "room": "development",
                "activity": activity,
                "subject": subject,
                "file_a": file_a,
                "file_b": file_b,
                "file_id_a": i,
                "file_id_b": i + 1,
                "path_a": str(path_a),
                "path_b": str(path_b),
            })

# Testing adjacent pairs
for room in TEST_ROOMS:
    for activity in ACTIVITIES:
        for subject in TEST_SUBJECTS:
            for i in range(1, 9):
                file_a = f"{i}.Csv"
                file_b = f"{i+1}.Csv"

                path_a = TEST_BASE_DIR / room / activity / subject / "CSV" / file_a
                path_b = TEST_BASE_DIR / room / activity / subject / "CSV" / file_b

                pair_records.append({
                    "split": "testing",
                    "room": room,
                    "activity": activity,
                    "subject": subject,
                    "file_a": file_a,
                    "file_b": file_b,
                    "file_id_a": i,
                    "file_id_b": i + 1,
                    "path_a": str(path_a),
                    "path_b": str(path_b),
                })

pairs_df = pd.DataFrame(pair_records)

print("===== ADJACENT PAIR LIST =====")
print(f"Total adjacent pairs: {len(pairs_df):,}")
display(pairs_df.head())

===== ADJACENT PAIR LIST =====
Total adjacent pairs: 704


,split,room,activity,subject,file_a,file_b,file_id_a,file_id_b,path_a,path_b
0,development,development,Bungkuk,Adelia,1.Csv,2.Csv,1,2,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/1.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv
1,development,development,Bungkuk,Adelia,2.Csv,3.Csv,2,3,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv
2,development,development,Bungkuk,Adelia,3.Csv,4.Csv,3,4,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv
3,development,development,Bungkuk,Adelia,4.Csv,5.Csv,4,5,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv
4,development,development,Bungkuk,Adelia,5.Csv,6.Csv,5,6,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/6.Csv


In [42]:
# Cell 6 - Fungsi load CSV dan hitung overlap

def classify_overlap(max_overlap: float) -> str:
    if pd.isna(max_overlap):
        return "ERROR"

    if max_overlap < TH_OK:
        return "OK"
    elif max_overlap < TH_WARNING:
        return "WARNING"
    elif max_overlap < TH_HIGH:
        return "HIGH_OVERLAP"
    else:
        return "CRITICAL_OVERLAP"


def load_required_csv(path: Path, required_columns):
    """
    Load CSV, normalize column names, keep required columns only.
    Minimal processing:
    - strip column names
    - numeric conversion
    - drop NA on required columns
    """
    if not path.exists():
        return None, f"MISSING_FILE: {path}"

    try:
        df = pd.read_csv(path)
    except Exception as e:
        return None, f"READ_ERROR: {repr(e)}"

    df.columns = [str(c).strip() for c in df.columns]

    missing_cols = [c for c in required_columns if c not in df.columns]
    if missing_cols:
        return None, f"MISSING_COLUMNS: {missing_cols}"

    df = df[required_columns].copy()

    for col in required_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before_drop = len(df)
    df = df.dropna(subset=required_columns).copy()
    after_drop = len(df)

    df.attrs["rows_before_dropna"] = before_drop
    df.attrs["rows_after_dropna"] = after_drop
    df.attrs["rows_dropped_na"] = before_drop - after_drop

    return df, None


def make_row_hash_set(df: pd.DataFrame, round_decimals_xyz: int = 6):
    """
    Hash unique rows based on Timestamp + X + Y + Z.
    X/Y/Z rounded for floating precision stability.
    """
    temp = df[["Timestamp", "X", "Y", "Z"]].copy()

    temp["X"] = temp["X"].round(round_decimals_xyz)
    temp["Y"] = temp["Y"].round(round_decimals_xyz)
    temp["Z"] = temp["Z"].round(round_decimals_xyz)

    row_hash = pd.util.hash_pandas_object(temp, index=False)
    return set(row_hash.values)


def compare_two_files(path_a, path_b):
    path_a = Path(path_a)
    path_b = Path(path_b)

    base_result = {
        "path_a": str(path_a),
        "path_b": str(path_b),

        "n_rows_a": np.nan,
        "n_rows_b": np.nan,
        "rows_before_dropna_a": np.nan,
        "rows_before_dropna_b": np.nan,
        "rows_dropped_na_a": np.nan,
        "rows_dropped_na_b": np.nan,

        "unique_timestamp_a": np.nan,
        "unique_timestamp_b": np.nan,
        "overlap_timestamp": np.nan,
        "overlap_ratio_a": np.nan,
        "overlap_ratio_b": np.nan,

        "unique_row_hash_a": np.nan,
        "unique_row_hash_b": np.nan,
        "identical_unique_row_hash": np.nan,
        "identical_ratio_a": np.nan,
        "identical_ratio_b": np.nan,

        "timestamp_min_a": np.nan,
        "timestamp_max_a": np.nan,
        "timestamp_min_b": np.nan,
        "timestamp_max_b": np.nan,
        "duration_a": np.nan,
        "duration_b": np.nan,

        "max_overlap_ratio": np.nan,
        "status_flag": "UNKNOWN",
        "error_message": "",
    }

    df_a, err_a = load_required_csv(path_a, REQUIRED_COLUMNS)
    df_b, err_b = load_required_csv(path_b, REQUIRED_COLUMNS)

    if err_a is not None:
        base_result["status_flag"] = "ERROR_A"
        base_result["error_message"] = err_a
        return base_result

    if err_b is not None:
        base_result["status_flag"] = "ERROR_B"
        base_result["error_message"] = err_b
        return base_result

    # Basic stats
    base_result["n_rows_a"] = len(df_a)
    base_result["n_rows_b"] = len(df_b)

    base_result["rows_before_dropna_a"] = df_a.attrs.get("rows_before_dropna", np.nan)
    base_result["rows_before_dropna_b"] = df_b.attrs.get("rows_before_dropna", np.nan)
    base_result["rows_dropped_na_a"] = df_a.attrs.get("rows_dropped_na", np.nan)
    base_result["rows_dropped_na_b"] = df_b.attrs.get("rows_dropped_na", np.nan)

    if len(df_a) == 0 or len(df_b) == 0:
        base_result["status_flag"] = "EMPTY_FILE"
        base_result["error_message"] = "One or both files are empty after dropna."
        return base_result

    base_result["timestamp_min_a"] = df_a["Timestamp"].min()
    base_result["timestamp_max_a"] = df_a["Timestamp"].max()
    base_result["timestamp_min_b"] = df_b["Timestamp"].min()
    base_result["timestamp_max_b"] = df_b["Timestamp"].max()

    base_result["duration_a"] = base_result["timestamp_max_a"] - base_result["timestamp_min_a"]
    base_result["duration_b"] = base_result["timestamp_max_b"] - base_result["timestamp_min_b"]

    # Timestamp overlap
    ts_a = set(df_a["Timestamp"].dropna().unique())
    ts_b = set(df_b["Timestamp"].dropna().unique())
    ts_inter = ts_a.intersection(ts_b)

    base_result["unique_timestamp_a"] = len(ts_a)
    base_result["unique_timestamp_b"] = len(ts_b)
    base_result["overlap_timestamp"] = len(ts_inter)
    base_result["overlap_ratio_a"] = len(ts_inter) / len(ts_a) if len(ts_a) else np.nan
    base_result["overlap_ratio_b"] = len(ts_inter) / len(ts_b) if len(ts_b) else np.nan

    # Identical row overlap based on Timestamp + X + Y + Z
    hash_a = make_row_hash_set(df_a, ROUND_DECIMALS_XYZ)
    hash_b = make_row_hash_set(df_b, ROUND_DECIMALS_XYZ)
    hash_inter = hash_a.intersection(hash_b)

    base_result["unique_row_hash_a"] = len(hash_a)
    base_result["unique_row_hash_b"] = len(hash_b)
    base_result["identical_unique_row_hash"] = len(hash_inter)
    base_result["identical_ratio_a"] = len(hash_inter) / len(hash_a) if len(hash_a) else np.nan
    base_result["identical_ratio_b"] = len(hash_inter) / len(hash_b) if len(hash_b) else np.nan

    max_overlap = np.nanmax([
        base_result["overlap_ratio_a"],
        base_result["overlap_ratio_b"],
        base_result["identical_ratio_a"],
        base_result["identical_ratio_b"],
    ])

    base_result["max_overlap_ratio"] = max_overlap
    base_result["status_flag"] = classify_overlap(max_overlap)

    return base_result

print("Functions ready.")

Functions ready.


In [43]:
# Cell 7 - Jalankan audit similarity seluruh pasangan file berurutan

audit_results = []

print("===== RUNNING GLOBAL ADJACENT PAIR SIMILARITY AUDIT =====")
print(f"Total pairs to audit: {len(pairs_df):,}")

for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df)):
    result = compare_two_files(row["path_a"], row["path_b"])

    # Add metadata
    result.update({
        "split": row["split"],
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "file_a": row["file_a"],
        "file_b": row["file_b"],
        "file_id_a": row["file_id_a"],
        "file_id_b": row["file_id_b"],
    })

    audit_results.append(result)

similarity_df = pd.DataFrame(audit_results)

# Reorder columns for readability
preferred_cols = [
    "split", "room", "activity", "subject",
    "file_a", "file_b", "file_id_a", "file_id_b",
    "status_flag", "max_overlap_ratio",

    "overlap_timestamp",
    "overlap_ratio_a", "overlap_ratio_b",

    "identical_unique_row_hash",
    "identical_ratio_a", "identical_ratio_b",

    "unique_timestamp_a", "unique_timestamp_b",
    "unique_row_hash_a", "unique_row_hash_b",

    "n_rows_a", "n_rows_b",
    "rows_before_dropna_a", "rows_before_dropna_b",
    "rows_dropped_na_a", "rows_dropped_na_b",

    "timestamp_min_a", "timestamp_max_a",
    "timestamp_min_b", "timestamp_max_b",
    "duration_a", "duration_b",

    "path_a", "path_b",
    "error_message",
]

similarity_df = similarity_df[[c for c in preferred_cols if c in similarity_df.columns]]

print("===== AUDIT DONE =====")
print(f"Total results: {len(similarity_df):,}")

display(similarity_df.head())

===== RUNNING GLOBAL ADJACENT PAIR SIMILARITY AUDIT =====
Total pairs to audit: 704


  0%|          | 0/704 [00:00<?, ?it/s]

/tmp/ipykernel_138220/581580184.py:29: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/tmp/ipykernel_138220/581580184.py:29: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/tmp/ipykernel_138220/581580184.py:29: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/tmp/ipykernel_138220/581580184.py:29: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/tmp/ipykernel_138220/581580184.py:29: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/tmp/ipykernel_138220/581580184.py:29: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/tmp/ipykernel_1

===== AUDIT DONE =====
Total results: 704


,split,room,activity,subject,file_a,file_b,file_id_a,file_id_b,status_flag,max_overlap_ratio,...,rows_dropped_na_b,timestamp_min_a,timestamp_max_a,timestamp_min_b,timestamp_max_b,duration_a,duration_b,path_a,path_b,error_message
0,development,development,Bungkuk,Adelia,1.Csv,2.Csv,1,2,OK,0.0,...,1,4619236332940,4624821312940,4630116512940,4635601492940,5584980000,5484980000,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/1.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,
1,development,development,Bungkuk,Adelia,2.Csv,3.Csv,2,3,OK,0.0,...,1,4630116512940,4635601492940,4644365872940,4650019252940,5484980000,5653380000,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,
2,development,development,Bungkuk,Adelia,3.Csv,4.Csv,3,4,OK,0.0,...,1,4644365872940,4650019252940,4659725892940,4665579192940,5653380000,5853300000,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv,
3,development,development,Bungkuk,Adelia,4.Csv,5.Csv,4,5,OK,0.0,...,1,4659725892940,4665579192940,4672341732940,4678027852940,5853300000,5686120000,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv,
4,development,development,Bungkuk,Adelia,5.Csv,6.Csv,5,6,OK,0.0,...,1,4672341732940,4678027852940,4693159832940,4699345292940,5686120000,6185460000,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/6.Csv,


In [44]:
# Cell 8 - Simpan hasil audit utama

all_pairs_path = AUDIT_OUT_DIR / "similarity_all_adjacent_pairs.csv"
similarity_df.to_csv(all_pairs_path, index=False)

problem_df = similarity_df[similarity_df["status_flag"] != "OK"].copy()
problem_pairs_path = AUDIT_OUT_DIR / "similarity_problem_pairs.csv"
problem_df.to_csv(problem_pairs_path, index=False)

print("===== SAVED AUDIT RESULTS =====")
print(f"All adjacent pairs : {all_pairs_path}")
print(f"Problem pairs      : {problem_pairs_path}")
print(f"Problem count      : {len(problem_df):,}")

display(problem_df.head(30))

===== SAVED AUDIT RESULTS =====
All adjacent pairs : /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/similarity_all_adjacent_pairs.csv
Problem pairs      : /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/similarity_problem_pairs.csv
Problem count      : 0


,split,room,activity,subject,file_a,file_b,file_id_a,file_id_b,status_flag,max_overlap_ratio,...,rows_dropped_na_b,timestamp_min_a,timestamp_max_a,timestamp_min_b,timestamp_max_b,duration_a,duration_b,path_a,path_b,error_message


In [45]:
# Cell 9 - Ringkasan status global

print("===== STATUS FLAG COUNTS =====")
status_counts = (
    similarity_df["status_flag"]
    .value_counts()
    .reset_index()
)

status_counts.columns = ["status_flag", "count"]
display(status_counts)

print("===== MAX OVERLAP SUMMARY =====")
display(similarity_df["max_overlap_ratio"].describe().to_frame("max_overlap_ratio"))

print("===== TOP 30 HIGHEST OVERLAP PAIRS =====")
display(
    similarity_df
    .sort_values("max_overlap_ratio", ascending=False)
    .head(30)
)

===== STATUS FLAG COUNTS =====


,status_flag,count
0,OK,704


===== MAX OVERLAP SUMMARY =====


,max_overlap_ratio
count,704.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


===== TOP 30 HIGHEST OVERLAP PAIRS =====


,split,room,activity,subject,file_a,file_b,file_id_a,file_id_b,status_flag,max_overlap_ratio,...,rows_dropped_na_b,timestamp_min_a,timestamp_max_a,timestamp_min_b,timestamp_max_b,duration_a,duration_b,path_a,path_b,error_message
0,development,development,Bungkuk,Adelia,1.Csv,2.Csv,1,2,OK,0.0,...,1,4619236332940,4624821312940,4630116512940,4635601492940,5584980000,5484980000,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/1.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,
1,development,development,Bungkuk,Adelia,2.Csv,3.Csv,2,3,OK,0.0,...,1,4630116512940,4635601492940,4644365872940,4650019252940,5484980000,5653380000,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,
464,testing,Controlled Room,Jongkok,Kanaya,1.Csv,2.Csv,1,2,OK,0.0,...,1,2995664192940,3001117032940,3005361812940,3011019472940,5452840000,5657660000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/1.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/2.Csv,
465,testing,Controlled Room,Jongkok,Kanaya,2.Csv,3.Csv,2,3,OK,0.0,...,1,3005361812940,3011019472940,3016732692940,3022751952940,5657660000,6019260000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/2.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/3.Csv,
466,testing,Controlled Room,Jongkok,Kanaya,3.Csv,4.Csv,3,4,OK,0.0,...,1,3016732692940,3022751952940,3033015732940,3038536292940,6019260000,5520560000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/3.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/4.Csv,
467,testing,Controlled Room,Jongkok,Kanaya,4.Csv,5.Csv,4,5,OK,0.0,...,1,3033015732940,3038536292940,3047940992940,3053661032940,5520560000,5720040000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/4.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/5.Csv,
468,testing,Controlled Room,Jongkok,Kanaya,5.Csv,6.Csv,5,6,OK,0.0,...,1,3047940992940,3053661032940,3063687112940,3069207112940,5720040000,5520000000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/5.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/6.Csv,
469,testing,Controlled Room,Jongkok,Kanaya,6.Csv,7.Csv,6,7,OK,0.0,...,1,3063687112940,3069207112940,3075697712940,3082451312940,5520000000,6753600000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/6.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/7.Csv,
470,testing,Controlled Room,Jongkok,Kanaya,7.Csv,8.Csv,7,8,OK,0.0,...,1,3075697712940,3082451312940,3088863632940,3094615952940,6753600000,5752320000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/7.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/8.Csv,
471,testing,Controlled Room,Jongkok,Kanaya,8.Csv,9.Csv,8,9,OK,0.0,...,1,3088863632940,3094615952940,3108521812940,3114308692940,5752320000,5786880000,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/8.Csv,/media/spell/Spell-lab/Lidar/A.Raw_Testing/Controlled Room/Jongkok/Kanaya/CSV/9.Csv,


In [46]:
# Cell 10 - Summary by split

def build_group_summary(df, group_cols):
    summary = (
        df.groupby(group_cols)
        .agg(
            n_pairs=("max_overlap_ratio", "count"),
            max_overlap=("max_overlap_ratio", "max"),
            mean_overlap=("max_overlap_ratio", "mean"),
            median_overlap=("max_overlap_ratio", "median"),
            n_ok=("status_flag", lambda x: int((x == "OK").sum())),
            n_warning=("status_flag", lambda x: int((x == "WARNING").sum())),
            n_high_overlap=("status_flag", lambda x: int((x == "HIGH_OVERLAP").sum())),
            n_critical_overlap=("status_flag", lambda x: int((x == "CRITICAL_OVERLAP").sum())),
            n_error=("status_flag", lambda x: int(x.astype(str).str.contains("ERROR").sum())),
        )
        .reset_index()
        .sort_values("max_overlap", ascending=False)
    )

    return summary

summary_by_split = build_group_summary(similarity_df, ["split"])

summary_by_split_path = AUDIT_OUT_DIR / "summary_by_split.csv"
summary_by_split.to_csv(summary_by_split_path, index=False)

print(f"Saved: {summary_by_split_path}")
display(summary_by_split)

Saved: /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/summary_by_split.csv


,split,n_pairs,max_overlap,mean_overlap,median_overlap,n_ok,n_warning,n_high_overlap,n_critical_overlap,n_error
0,development,384,0.0,0.0,0.0,384,0,0,0,0
1,testing,320,0.0,0.0,0.0,320,0,0,0,0


In [47]:
# Cell 11 - Summary by activity

summary_by_activity = build_group_summary(similarity_df, ["split", "activity"])

summary_by_activity_path = AUDIT_OUT_DIR / "summary_by_activity.csv"
summary_by_activity.to_csv(summary_by_activity_path, index=False)

print(f"Saved: {summary_by_activity_path}")
display(summary_by_activity)

Saved: /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/summary_by_activity.csv


,split,activity,n_pairs,max_overlap,mean_overlap,median_overlap,n_ok,n_warning,n_high_overlap,n_critical_overlap,n_error
0,development,Bungkuk,96,0.0,0.0,0.0,96,0,0,0,0
1,development,Duduk,96,0.0,0.0,0.0,96,0,0,0,0
2,development,Jatuh,96,0.0,0.0,0.0,96,0,0,0,0
3,development,Jongkok,96,0.0,0.0,0.0,96,0,0,0,0
4,testing,Bungkuk,80,0.0,0.0,0.0,80,0,0,0,0
5,testing,Duduk,80,0.0,0.0,0.0,80,0,0,0,0
6,testing,Jatuh,80,0.0,0.0,0.0,80,0,0,0,0
7,testing,Jongkok,80,0.0,0.0,0.0,80,0,0,0,0


In [48]:
# Cell 12 - Summary by subject

summary_by_subject = build_group_summary(similarity_df, ["split", "room", "activity", "subject"])

summary_by_subject_path = AUDIT_OUT_DIR / "summary_by_subject.csv"
summary_by_subject.to_csv(summary_by_subject_path, index=False)

print(f"Saved: {summary_by_subject_path}")
display(summary_by_subject.head(50))

Saved: /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/summary_by_subject.csv


,split,room,activity,subject,n_pairs,max_overlap,mean_overlap,median_overlap,n_ok,n_warning,n_high_overlap,n_critical_overlap,n_error
0,development,development,Bungkuk,Adelia,8,0.0,0.0,0.0,8,0,0,0,0
1,development,development,Bungkuk,Afi,8,0.0,0.0,0.0,8,0,0,0,0
64,testing,Controlled Room,Jongkok,Naila,8,0.0,0.0,0.0,8,0,0,0,0
63,testing,Controlled Room,Jongkok,Kanaya,8,0.0,0.0,0.0,8,0,0,0,0
62,testing,Controlled Room,Jatuh,Zaira,8,0.0,0.0,0.0,8,0,0,0,0
61,testing,Controlled Room,Jatuh,Rega,8,0.0,0.0,0.0,8,0,0,0,0
60,testing,Controlled Room,Jatuh,Nana,8,0.0,0.0,0.0,8,0,0,0,0
59,testing,Controlled Room,Jatuh,Naila,8,0.0,0.0,0.0,8,0,0,0,0
58,testing,Controlled Room,Jatuh,Kanaya,8,0.0,0.0,0.0,8,0,0,0,0
57,testing,Controlled Room,Duduk,Zaira,8,0.0,0.0,0.0,8,0,0,0,0


In [49]:
# Cell 13 - Summary by room

summary_by_room = build_group_summary(similarity_df, ["split", "room"])

summary_by_room_path = AUDIT_OUT_DIR / "summary_by_room.csv"
summary_by_room.to_csv(summary_by_room_path, index=False)

print(f"Saved: {summary_by_room_path}")
display(summary_by_room)

Saved: /media/spell/Spell-lab/Lidar/A.Raw Dataset/audit_similarity dataset/summary_by_room.csv


,split,room,n_pairs,max_overlap,mean_overlap,median_overlap,n_ok,n_warning,n_high_overlap,n_critical_overlap,n_error
0,development,development,384,0.0,0.0,0.0,384,0,0,0,0
1,testing,Controlled Room,160,0.0,0.0,0.0,160,0,0,0,0
2,testing,Uncontrolled Room,160,0.0,0.0,0.0,160,0,0,0,0


In [50]:
# Cell 14 - Tampilkan pasangan bermasalah paling serius

critical_df = similarity_df[similarity_df["status_flag"] == "CRITICAL_OVERLAP"].copy()
high_df = similarity_df[similarity_df["status_flag"] == "HIGH_OVERLAP"].copy()
warning_df = similarity_df[similarity_df["status_flag"] == "WARNING"].copy()

print("===== PROBLEM SUMMARY =====")
print(f"CRITICAL_OVERLAP : {len(critical_df):,}")
print(f"HIGH_OVERLAP     : {len(high_df):,}")
print(f"WARNING          : {len(warning_df):,}")

print("\n===== CRITICAL OVERLAP PAIRS =====")
display(
    critical_df
    .sort_values("max_overlap_ratio", ascending=False)
    .head(50)
)

print("\n===== HIGH OVERLAP PAIRS =====")
display(
    high_df
    .sort_values("max_overlap_ratio", ascending=False)
    .head(50)
)

print("\n===== WARNING PAIRS =====")
display(
    warning_df
    .sort_values("max_overlap_ratio", ascending=False)
    .head(50)
)

===== PROBLEM SUMMARY =====
CRITICAL_OVERLAP : 0
HIGH_OVERLAP     : 0

===== CRITICAL OVERLAP PAIRS =====


,split,room,activity,subject,file_a,file_b,file_id_a,file_id_b,status_flag,max_overlap_ratio,...,rows_dropped_na_b,timestamp_min_a,timestamp_max_a,timestamp_min_b,timestamp_max_b,duration_a,duration_b,path_a,path_b,error_message



===== HIGH OVERLAP PAIRS =====


,split,room,activity,subject,file_a,file_b,file_id_a,file_id_b,status_flag,max_overlap_ratio,...,rows_dropped_na_b,timestamp_min_a,timestamp_max_a,timestamp_min_b,timestamp_max_b,duration_a,duration_b,path_a,path_b,error_message



===== WARNING PAIRS =====


,split,room,activity,subject,file_a,file_b,file_id_a,file_id_b,status_flag,max_overlap_ratio,...,rows_dropped_na_b,timestamp_min_a,timestamp_max_a,timestamp_min_b,timestamp_max_b,duration_a,duration_b,path_a,path_b,error_message


In [51]:
# Cell 15 - Sanity check pasangan tertentu secara manual

def inspect_pair(split, activity, subject, file_a, file_b, room=None):
    temp = similarity_df.copy()

    mask = (
        (temp["split"] == split) &
        (temp["activity"] == activity) &
        (temp["subject"] == subject) &
        (temp["file_a"] == file_a) &
        (temp["file_b"] == file_b)
    )

    if room is not None:
        mask &= (temp["room"] == room)

    result = temp[mask].copy()

    if len(result) == 0:
        print("Pair not found.")
        return None

    display(result.T)
    return result

# Contoh development:
# inspect_pair("development", "Bungkuk", "Afi", "1.Csv", "2.Csv", room="development")

# Contoh testing:
# inspect_pair("testing", "Bungkuk", "Kanaya", "1.Csv", "2.Csv", room="Controlled Room")

In [52]:
# Cell 16 - Interpretasi otomatis sederhana

total_pairs = len(similarity_df)
ok_count = int((similarity_df["status_flag"] == "OK").sum())
problem_count = total_pairs - ok_count

critical_count = int((similarity_df["status_flag"] == "CRITICAL_OVERLAP").sum())
high_count = int((similarity_df["status_flag"] == "HIGH_OVERLAP").sum())
warning_count = int((similarity_df["status_flag"] == "WARNING").sum())

max_overlap_global = similarity_df["max_overlap_ratio"].max()
mean_overlap_global = similarity_df["max_overlap_ratio"].mean()

print("===== SIMPLE GLOBAL INTERPRETATION =====")
print(f"Total adjacent pairs audited : {total_pairs:,}")
print(f"OK pairs                     : {ok_count:,}")
print(f"Problem pairs                : {problem_count:,}")
print(f"WARNING                      : {warning_count:,}")
print(f"HIGH_OVERLAP                 : {high_count:,}")
print(f"CRITICAL_OVERLAP             : {critical_count:,}")
print(f"Global max overlap           : {max_overlap_global:.6f}")
print(f"Global mean overlap          : {mean_overlap_global:.6f}")

print("\nInterpretation:")

if critical_count > 0:
    print("- Ada pasangan dengan CRITICAL_OVERLAP. File terkait perlu dicek dan kemungkinan perlu convert ulang.")
elif high_count > 0:
    print("- Tidak ada critical overlap, tetapi ada HIGH_OVERLAP. File terkait sebaiknya dicek.")
elif warning_count > 0:
    print("- Mayoritas aman, tetapi ada WARNING. Cek ringan pasangan tersebut.")
else:
    print("- Semua adjacent pairs berstatus OK. Dataset hasil re-convert tampak bersih dari overlap antar file berurutan.")

print("\nImportant note:")
print("- Audit ini mengecek overlap timestamp dan identical row berdasarkan Timestamp + X + Y + Z.")
print("- Audit ini belum mengecek kualitas gerakan, ROI, GCZT, DBSCAN, atau label aktivitas.")
print("- Jika hasil ini OK, lanjutkan ke cleaning dan frame builder.")

===== SIMPLE GLOBAL INTERPRETATION =====
Total adjacent pairs audited : 704
OK pairs                     : 704
Problem pairs                : 0
HIGH_OVERLAP                 : 0
CRITICAL_OVERLAP             : 0
Global max overlap           : 0.000000
Global mean overlap          : 0.000000

Interpretation:
- Semua adjacent pairs berstatus OK. Dataset hasil re-convert tampak bersih dari overlap antar file berurutan.

Important note:
- Audit ini mengecek overlap timestamp dan identical row berdasarkan Timestamp + X + Y + Z.
- Audit ini belum mengecek kualitas gerakan, ROI, GCZT, DBSCAN, atau label aktivitas.
- Jika hasil ini OK, lanjutkan ke cleaning dan frame builder.


In [53]:
# Debug 1 - Lihat error_message dari hasil audit

pd.set_option("display.max_colwidth", 200)

display(
    similarity_df[
        ["split", "room", "activity", "subject", "file_a", "path_a", "status_flag", "error_message"]
    ]
    .head(20)
)

,split,room,activity,subject,file_a,path_a,status_flag,error_message
0,development,development,Bungkuk,Adelia,1.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/1.Csv,OK,
1,development,development,Bungkuk,Adelia,2.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,OK,
2,development,development,Bungkuk,Adelia,3.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,OK,
3,development,development,Bungkuk,Adelia,4.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv,OK,
4,development,development,Bungkuk,Adelia,5.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv,OK,
5,development,development,Bungkuk,Adelia,6.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/6.Csv,OK,
6,development,development,Bungkuk,Adelia,7.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/7.Csv,OK,
7,development,development,Bungkuk,Adelia,8.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/8.Csv,OK,
8,development,development,Bungkuk,Afi,1.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Afi/CSV/1.Csv,OK,
9,development,development,Bungkuk,Afi,2.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Afi/CSV/2.Csv,OK,


In [54]:
# Debug 2 - Ringkasan manifest file

print("Total expected files:", len(manifest_df))
print("Existing files:", manifest_df["exists"].sum())
print("Missing files:", (~manifest_df["exists"]).sum())

display(
    manifest_df["exists"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "exists", "exists": "count"})
)

display(manifest_df.head(20))
display(manifest_df[manifest_df["exists"] == False].head(20))

Total expected files: 792
Existing files: 792
Missing files: 0


,count,count
0,True,792


,split,room,activity,subject,file_name,file_id,file_path,exists
0,development,development,Bungkuk,Adelia,1.Csv,1,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/1.Csv,True
1,development,development,Bungkuk,Adelia,2.Csv,2,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,True
2,development,development,Bungkuk,Adelia,3.Csv,3,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,True
3,development,development,Bungkuk,Adelia,4.Csv,4,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv,True
4,development,development,Bungkuk,Adelia,5.Csv,5,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv,True
5,development,development,Bungkuk,Adelia,6.Csv,6,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/6.Csv,True
6,development,development,Bungkuk,Adelia,7.Csv,7,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/7.Csv,True
7,development,development,Bungkuk,Adelia,8.Csv,8,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/8.Csv,True
8,development,development,Bungkuk,Adelia,9.Csv,9,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/9.Csv,True
9,development,development,Bungkuk,Afi,1.Csv,1,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Afi/CSV/1.Csv,True


,split,room,activity,subject,file_name,file_id,file_path,exists


In [55]:
pd.set_option("display.max_colwidth", 300)

display(
    similarity_df[
        ["split", "room", "activity", "subject", "file_a", "path_a", "status_flag", "error_message"]
    ].head(20)
)

,split,room,activity,subject,file_a,path_a,status_flag,error_message
0,development,development,Bungkuk,Adelia,1.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/1.Csv,OK,
1,development,development,Bungkuk,Adelia,2.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/2.Csv,OK,
2,development,development,Bungkuk,Adelia,3.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/3.Csv,OK,
3,development,development,Bungkuk,Adelia,4.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/4.Csv,OK,
4,development,development,Bungkuk,Adelia,5.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/5.Csv,OK,
5,development,development,Bungkuk,Adelia,6.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/6.Csv,OK,
6,development,development,Bungkuk,Adelia,7.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/7.Csv,OK,
7,development,development,Bungkuk,Adelia,8.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Adelia/CSV/8.Csv,OK,
8,development,development,Bungkuk,Afi,1.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Afi/CSV/1.Csv,OK,
9,development,development,Bungkuk,Afi,2.Csv,/media/spell/Spell-lab/Lidar/A.Raw Dataset/Bungkuk/Afi/CSV/2.Csv,OK,


In [56]:
print("DEV_BASE_DIR exists:", DEV_BASE_DIR.exists())
print("TEST_BASE_DIR exists:", TEST_BASE_DIR.exists())

print("\nSample .Csv files in DEV:")
for p in list(DEV_BASE_DIR.rglob("*.Csv"))[:20]:
    print(p)

print("\nSample .csv files in DEV:")
for p in list(DEV_BASE_DIR.rglob("*.csv"))[:20]:
    print(p)

print("\nSample .Csv files in TEST:")
for p in list(TEST_BASE_DIR.rglob("*.Csv"))[:20]:
    print(p)

print("\nSample .csv files in TEST:")
for p in list(TEST_BASE_DIR.rglob("*.csv"))[:20]:
    print(p)

DEV_BASE_DIR exists: True
TEST_BASE_DIR exists: True

Sample .Csv files in DEV:
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/4.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/7.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/6.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/5.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/9.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/1.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/3.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/2.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Afi/CSV/8.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Teguh/CSV/4.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Teguh/CSV/7.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Teguh/CSV/6.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Teguh/CSV/5.Csv
/media/spell/Spell-lab/Lidar/A.Raw Dataset/Jongkok/Teguh/CSV/9.Csv
